# 🧮 Statistik Penelitian: Uji ANOVA (ala SPSS)

Analisis statistik lengkap untuk skripsi — langsung di notebook dengan
**pingouin**: uji normalitas, homogenitas, ANOVA satu arah, *post-hoc*, dan
*effect size*. Output berupa tabel rapi yang siap disalin ke laporan.

**Yang kamu pelajari:** alur uji hipotesis yang benar + visualisasi publikasi.

In [ ]:
# Sel 1 — data contoh: nilai ujian 3 kelompok metode belajar.
# Ganti dengan datamu:  df = pd.read_excel("data.xlsx")
import numpy as np, pandas as pd, pingouin as pg

rng = np.random.default_rng(7)
df = pd.DataFrame({
    "metode": ["Ceramah"] * 30 + ["Diskusi"] * 30 + ["Praktikum"] * 30,
    "nilai": np.concatenate([
        rng.normal(70, 8, 30),   # Ceramah
        rng.normal(75, 8, 30),   # Diskusi
        rng.normal(82, 8, 30),   # Praktikum
    ]).round(1),
})
df.groupby("metode")["nilai"].describe().round(2)

In [ ]:
# Sel 2 — uji asumsi: normalitas per kelompok (Shapiro-Wilk) & homogenitas (Levene).
print("Normalitas (p > 0.05 = normal):")
display(pg.normality(df, dv="nilai", group="metode").round(4))
print("Homogenitas varians (p > 0.05 = homogen):")
display(pg.homoscedasticity(df, dv="nilai", group="metode").round(4))

In [ ]:
# Sel 3 — ANOVA satu arah + effect size (eta-squared).
anova = pg.anova(df, dv="nilai", between="metode", detailed=True)
display(anova.round(4))
p = anova.loc[0, "p-unc"]
print(f"Kesimpulan: {'ADA' if p < 0.05 else 'TIDAK ada'} perbedaan signifikan antar metode (p={p:.4f}).")

In [ ]:
# Sel 4 — post-hoc Tukey HSD: pasangan mana yang berbeda?
pg.pairwise_tukey(df, dv="nilai", between="metode").round(4)

In [ ]:
# Sel 5 — visual publikasi: boxplot + titik data.
import matplotlib.pyplot as plt, seaborn as sns

fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=df, x="metode", y="nilai", hue="metode", palette="Set2", ax=ax, legend=False)
sns.stripplot(data=df, x="metode", y="nilai", color="0.25", size=3, ax=ax)
ax.set_title("Perbandingan nilai per metode belajar")
plt.tight_layout()

**Langkah lanjut:** data tidak normal? pakai `pg.kruskal` (non-parametrik).
Dua kelompok saja? `pg.ttest`. Ada 2 faktor? `pg.anova(between=["a","b"])`.
Korelasi & regresi juga ada: `pg.corr`, `pg.linear_regression`.